# France Food Delivery Prospecting

This notebook explores OpenStreetMap business data for France and prepares a prospect list for a new food-delivery app. It filters the source to restaurants, fast-food venues, bakeries, bars, and cafes; reviews contact coverage, possible duplicates, data freshness, and geographic distribution; then prepares and exports the lead list. Run `france_geofabrik_raw_data/clean_geofabrik_food_venues.ipynb` first to prepare the Geofabrik input.

**Official source sites:** [BANCO — Base Nationale des Commerces Ouverte](https://www.data.gouv.fr/datasets/base-nationale-des-commerces-ouverte) and [Geofabrik — OpenStreetMap France extract](https://download.geofabrik.de/europe/france.html).

**Raw data:** France-wide business records from BANCO are stored in `france_banco_raw_data/data.csv`; source metadata and licensing details are listed in `france_banco_raw_data/metadata.csv` and `france_banco_raw_data/license.txt`. The raw OpenStreetMap-derived food-venue extract and cuisine mapping are in `france_geofabrik_raw_data/`; the preparation notebook writes `food_venues_cleaned.csv`, which this analysis loads. The compact INSEE 2023 commune population lookup used in section 3 is `insee_commune_population_2023/commune_population_2023.csv`, extracted from the [INSEE population tables](https://www.insee.fr/fr/statistiques/9006235). Section 4 uses `commune_region_2026.csv`, a compact commune-to-region lookup derived from the [INSEE 2026 Code officiel géographique](https://www.insee.fr/fr/information/8740222). Records and contact details may be incomplete or out of date, so verify prospects before outreach.

**Output:** `france_food_delivery_prospects.csv`

## 1. Filter and enrich the food-venue data

Filter the BANCO records to the target venue types, then enrich them with PBF tags by matching on `osm_id`. Keep all selected BANCO records and any unmatched PBF food venues. Shared fields prefer BANCO values and use PBF values only to fill missing entries.

### 1.1 Load the source files

Read the semicolon-delimited BANCO and Geofabrik extracts, keeping OSM IDs as strings for a safe merge.

In [ ]:
import pandas as pd

data_df = pd.read_csv(
    "france_banco_raw_data/data.csv", sep=";", low_memory=False, dtype={"osm_id": "string", "com_insee": "string"}
)
pbf_df = pd.read_csv(
    "france_geofabrik_raw_data/food_venues_cleaned.csv",
    sep=";", low_memory=False, dtype={"osm_id": "string", "com_insee": "string"}
)

print(f"BANCO: {len(data_df):,} rows, {len(data_df.columns)} columns")
print(f"Cleaned Geofabrik food venues: {len(pbf_df):,} rows, {len(pbf_df.columns)} columns")
available_pbf_columns = pbf_df.columns.to_list()


### 1.2 Review BANCO food-venue categories

Count the BANCO records in the target food-venue categories for reference. Section 1.4 enriches the full BANCO dataset; the prospect list is filtered to these categories later in section 2.


In [ ]:
venue_types = ["restaurant", "fast_food", "bakery", "bar", "cafe"]
filtered_data = data_df[data_df["type"].isin(venue_types)].copy()
print(f"Selected BANCO records: {len(filtered_data):,}")
banco_venue_type_counts = filtered_data["type"].value_counts()
# Show the Geofabrik food-venue records by type, as above for BANCO.
print(f"Geofabrik food-venue records: {len(pbf_df):,}")
geofabrik_venue_type_counts = pbf_df["type"].value_counts()

### 1.3 Prepare the Geofabrik data

The preparation notebook [`france_geofabrik_raw_data/clean_geofabrik_food_venues.ipynb`](france_geofabrik_raw_data/clean_geofabrik_food_venues.ipynb) inspects OSM tags, fills blank Geofabrik fields from those tags, extracts raw cuisine values, splits cuisine tags only on semicolons, normalizes tokens for matching, and maps them with `cuisine_mapping_v2.csv`. It keeps mapped cuisines and dishes in separate columns, matches cuisine labels to `cuisine_region_mapping.csv` and builds a deduplicated regional summary, drops the raw `tags` column, and saves `food_venues_cleaned.csv`. The token audit code is optional and commented out. Run the cleaner first; this notebook loads its saved venue-level output.


### 1.4 Enrich BANCO with cleaned Geofabrik fields

Use the complete BANCO `data.csv` as the master table and left-join the cleaned Geofabrik food-venue data on `osm_id`. Keep every BANCO row, validate that each source has one row per non-empty OSM ID, and fill blank shared fields from Geofabrik. Report Geofabrik IDs with no BANCO match separately; they are not included in this master-table merge.


In [ ]:
for source_name, source_df in (("BANCO data", data_df), ("cleaned Geofabrik food venues", pbf_df)):
    if source_df["osm_id"].isna().any() or source_df["osm_id"].astype("string").str.strip().eq("").any():
        raise ValueError(f"{source_name} contains rows without osm_id; cannot safely merge.")
    if source_df["osm_id"].duplicated().any():
        raise ValueError(f"{source_name} contains duplicate osm_id values; resolve them before merging.")

shared_columns = (set(data_df.columns) & set(pbf_df.columns)) - {"osm_id"}
df = data_df.merge(
    pbf_df,
    on="osm_id",
    how="left",
    suffixes=("_banco", "_pbf"),
    indicator="_source_match",
    validate="one_to_one",
)

for column in shared_columns:
    df[column] = df[f"{column}_banco"].combine_first(df[f"{column}_pbf"])
df = df.drop(columns=[f"{column}_{suffix}" for column in shared_columns for suffix in ("banco", "pbf")])

geofabrik_only_count = int((~pbf_df["osm_id"].isin(data_df["osm_id"])).sum())
match_counts = df["_source_match"].value_counts().rename(index={
    "both": "Matched in BANCO and Geofabrik",
    "left_only": "BANCO only; no Geofabrik match",
}).rename("records").to_frame()
matched_record_count = int((df["_source_match"] == "both").sum())
banco_only_record_count = int((df["_source_match"] == "left_only").sum())
print(f"BANCO records: {len(df):,}; matched Geofabrik records: {matched_record_count:,}; no Geofabrik match: {banco_only_record_count:,}")
print(f"Geofabrik OSM IDs without a BANCO match (not included in df): {geofabrik_only_count:,}")
cuisine_columns = [column for column in ["cuisine_raw", "cuisine_standard", "dish"] if column in df.columns]
print(f"Cuisine and dish columns included: {cuisine_columns}")


### 1.5 Filter the merged data to prospecting venue types

Select restaurants, fast-food venues, bakeries, bars, and cafes for the prospect analysis. Confirm the row counts and category totals, then inspect a concise sample of the selected records.


In [ ]:
food_types = ["restaurant", "fast_food", "bakery", "bar", "cafe"]
food_prospects = df.loc[df["type"].isin(food_types)].copy()
food_prospects = food_prospects.drop(
    columns=["level", "profession_ref", "wikidata", "wheelchair", "facebook"],
    errors="ignore",
)

# Trim names and contact fields once; treat empty and whitespace-only strings as missing.
text_fields = [column for column in ["name", "phone", "email", "website"] if column in food_prospects]
for column in text_fields:
    values = food_prospects[column].astype("string").str.strip()
    food_prospects[column] = values.mask(values.eq(""), pd.NA)

print(f"Rows in merged dataframe: {len(df):,}")
print(f"Rows after prospecting type filter: {len(food_prospects):,}")
print(f"Included types: {', '.join(food_types)}")
type_counts = food_prospects["type"].value_counts().reindex(food_types, fill_value=0)
prospects_by_type = type_counts.copy()

prospect_preview_columns = [
    column for column in [
        "osm_id", "name", "brand", "type", "cuisine_raw",
        "cuisine_standard", "dish", "com_nom", "address", "phone", "email", "website",
    ]
    if column in food_prospects.columns
]


## 2. France-wide prospect analysis

The analysis below uses `food_prospects`, the merged and target-type-filtered dataframe prepared in section 1.5.


### 2.1 France overview

Start with target-venue and cuisine-tag coverage, then compare the most common standardized cuisines. Cuisine shares are non-exclusive because venues can have multiple cuisine labels.

In [ ]:
import matplotlib.pyplot as plt

overview_cuisine_tokens = food_prospects[["osm_id", "cuisine_standard"]].copy()
overview_cuisine_tokens["canonical_cuisine"] = overview_cuisine_tokens["cuisine_standard"].astype("string").str.split(";")
overview_cuisine_tokens = overview_cuisine_tokens.explode("canonical_cuisine", ignore_index=True)
overview_cuisine_tokens["canonical_cuisine"] = (
    overview_cuisine_tokens["canonical_cuisine"].astype("string").str.normalize("NFKC").str.strip().str.casefold()
)
overview_cuisine_tokens = overview_cuisine_tokens.loc[
    overview_cuisine_tokens["canonical_cuisine"].notna()
    & overview_cuisine_tokens["canonical_cuisine"].ne(""),
    ["osm_id", "canonical_cuisine"],
].drop_duplicates(["osm_id", "canonical_cuisine"])

cuisine_counts_france = overview_cuisine_tokens.groupby("canonical_cuisine")["osm_id"].nunique().sort_values(ascending=False)
target_venue_count = food_prospects["osm_id"].nunique()
cuisine_tagged_venue_count = overview_cuisine_tokens["osm_id"].nunique()
cuisine_tag_coverage_pct = cuisine_tagged_venue_count / target_venue_count * 100 if target_venue_count else 0
distinct_cuisine_count = overview_cuisine_tokens["canonical_cuisine"].nunique()

overview_fig = plt.figure(figsize=(14, 9), constrained_layout=True)
overview_grid = overview_fig.add_gridspec(2, 1, height_ratios=[0.75, 3], hspace=0.25)
kpi_grid = overview_grid[0].subgridspec(1, 4, wspace=0.15)
kpis = [
    ("Target venues", f"{target_venue_count:,}"),
    ("Venues with cuisine tags", f"{cuisine_tagged_venue_count:,}"),
    ("Cuisine-tag coverage", f"{cuisine_tag_coverage_pct:.1f}%"),
    ("Distinct standardized cuisines", f"{distinct_cuisine_count:,}"),
]
for index, (label, value) in enumerate(kpis):
    kpi_ax = overview_fig.add_subplot(kpi_grid[0, index])
    kpi_ax.set_facecolor("#f1f5f9")
    kpi_ax.text(0.5, 0.65, value, ha="center", va="center", fontsize=20, fontweight="bold", color="#17324d")
    kpi_ax.text(0.5, 0.25, label, ha="center", va="center", fontsize=10, color="#334155", wrap=True)
    kpi_ax.set_xticks([])
    kpi_ax.set_yticks([])
    for spine in kpi_ax.spines.values():
        spine.set_visible(False)

top_cuisine_counts = cuisine_counts_france.head(15).sort_values()
cuisine_ax = overview_fig.add_subplot(overview_grid[1])
top_cuisine_counts.plot(kind="barh", ax=cuisine_ax, color="#367c8b")
cuisine_ax.set(
    title="15 most common standardized cuisines across France",
    xlabel="Distinct venues (non-exclusive counts)",
    ylabel="Canonical cuisine",
)
cuisine_ax.grid(axis="x", alpha=0.25)
overview_fig.suptitle("France-wide prospect and cuisine overview", fontsize=16, fontweight="bold")
plt.show()

Show missing-value counts and percentages across filtered prospects.


In [ ]:
missingness_data = food_prospects.drop(columns=["_source_match"], errors="ignore")
missing_mask = missingness_data.isna()
for column in missingness_data.select_dtypes(include=["object", "string"]).columns:
    blank_values = missingness_data[column].astype("string").str.strip().eq("").fillna(False)
    missing_mask[column] = missing_mask[column] | blank_values

missing_pct = missing_mask.mean().mul(100).sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_records": missing_mask.sum(),
    "missing_percent": missing_pct.round(1),
})
# missing_summary remains available for review without rendering a wide table.


Plot prospect counts by venue type.


In [ ]:
import matplotlib.pyplot as plt

category_counts = food_prospects["type"].value_counts().reindex(food_types, fill_value=0)
ax = category_counts.plot(kind="bar", color="steelblue", figsize=(9, 5))
ax.set(title="Prospects by venue type", xlabel="Venue type", ylabel="Number of prospects")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

Report the number of columns in the filtered prospect data.


In [ ]:
remaining_column_count = food_prospects.shape[1]

Count prospects with a name, contact details, or both.


In [ ]:
contact_fields = ["phone", "email", "website"]
has_contact = food_prospects[contact_fields].notna().any(axis=1)
has_name = food_prospects["name"].notna()

summary = pd.Series({
    "total prospects": len(food_prospects),
    "with a name": has_name.sum(),
    "with a phone, email, or website": has_contact.sum(),
    "with both a name and contact detail": (has_name & has_contact).sum(),
})

contact_summary = summary.copy()


Summarize venue and contact coverage by type and INSEE-coded commune.


In [ ]:
commune_code = food_prospects["com_insee"].astype("string").str.strip().replace("", pd.NA)
commune_name = food_prospects["com_nom"].astype("string").str.strip().replace("", pd.NA)
coverage_data = food_prospects.assign(
    has_name=has_name,
    has_contact=has_contact,
    commune_code=commune_code.fillna("Unknown"),
    commune_name=commune_name,
)
coverage_data["commune_label"] = "Unknown commune"
has_commune_code = coverage_data["commune_code"].ne("Unknown")
coverage_data.loc[has_commune_code, "commune_label"] = (
    coverage_data.loc[has_commune_code, "commune_name"].fillna("Unknown commune")
    + " [" + coverage_data.loc[has_commune_code, "commune_code"] + "]"
)
coverage_data["name_and_contact"] = coverage_data["has_name"] & coverage_data["has_contact"]

coverage_by_type = coverage_data.groupby("type").agg(
    prospects=("type", "size"),
    named=("has_name", "sum"),
    contactable=("has_contact", "sum"),
    named_and_contactable=("name_and_contact", "sum"),
)
coverage_by_type["contactable_%"] = (coverage_by_type["contactable"] / coverage_by_type["prospects"] * 100).round(1)
coverage_by_type = coverage_by_type.sort_values("prospects", ascending=False)

coverage_by_commune = coverage_data.groupby("commune_code", as_index=False, dropna=False).agg(
    commune_label=("commune_label", "first"),
    commune_name=("commune_name", "first"),
    prospects=("type", "size"),
    named=("has_name", "sum"),
    contactable=("has_contact", "sum"),
    named_and_contactable=("name_and_contact", "sum"),
)
coverage_by_commune["contactable_%"] = (
    coverage_by_commune["contactable"] / coverage_by_commune["prospects"] * 100
).round(1)
coverage_by_commune = coverage_by_commune.set_index("commune_label")
top_commune_contact_summary = coverage_by_commune.sort_values("prospects", ascending=False).head(20).copy()
known_communes = coverage_by_commune.loc[coverage_by_commune["commune_code"].ne("Unknown")]


Compare contactable counts by venue type and contact rates in the 20 largest communes.


In [ ]:
import matplotlib.pyplot as plt

top_communes = known_communes.sort_values("prospects", ascending=False).head(20)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

coverage_by_type.sort_values("contactable", ascending=False)["contactable"].plot(
    kind="bar", ax=axes[0], color="seagreen"
)
axes[0].set(title="Contactable prospects by venue type", xlabel="Venue type", ylabel="Prospects with contact details")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.25)

communes_by_contact_rate = top_communes.sort_values("contactable_%")
communes_by_contact_rate["contactable_%"].plot(
    kind="barh", ax=axes[1], color="darkorange"
)
axes[1].set_yticks(range(len(communes_by_contact_rate)))
axes[1].set_yticklabels(communes_by_contact_rate["commune_name"].fillna("Unknown commune"))
axes[1].set(title="Contact coverage in the 20 largest communes", xlabel="Prospects with phone, email, or website (%)", ylabel="Commune")
axes[1].grid(axis="x", alpha=0.25)

fig.tight_layout()
plt.show()


Rank the 20 known communes with the most prospects.


In [ ]:
import matplotlib.pyplot as plt

commune_count_plot = known_communes.nlargest(20, "prospects").sort_values("prospects")
ax = commune_count_plot["prospects"].plot(kind="barh", figsize=(10, 8), color="teal")
ax.set_yticks(range(len(commune_count_plot)))
ax.set_yticklabels(commune_count_plot["commune_name"].fillna("Unknown commune"))
ax.set(title="Prospects in the 20 communes with the most venues", xlabel="Number of prospects", ylabel="Commune")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


Show venue-type counts in the 10 known communes with the most prospects.


In [ ]:
import matplotlib.pyplot as plt

top_commune_codes = known_communes.nlargest(10, "prospects")["commune_code"]
commune_type_counts = (
    coverage_data[coverage_data["commune_code"].isin(top_commune_codes)]
    .groupby(["commune_code", "type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=food_types, fill_value=0)
)
commune_type_counts = commune_type_counts.loc[commune_type_counts.sum(axis=1).sort_values().index]
commune_type_names = (
    coverage_by_commune.set_index("commune_code").loc[commune_type_counts.index, "commune_name"]
    .fillna("Unknown commune")
)

ax = commune_type_counts.plot(kind="barh", stacked=True, figsize=(12, 7), colormap="tab20c")
ax.set_yticks(range(len(commune_type_counts)))
ax.set_yticklabels(commune_type_names)
ax.set(title="Prospect venue mix in the 10 largest communes", xlabel="Number of prospects", ylabel="Commune")
ax.legend(title="Venue type", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


Compare contact coverage in known communes with at least 500 prospects.


In [ ]:
import matplotlib.pyplot as plt

minimum_commune_prospects = 500
communes_for_contact_rate = known_communes[
    known_communes["prospects"] >= minimum_commune_prospects
].sort_values("contactable_%")

ax = communes_for_contact_rate["contactable_%"].plot(
    kind="barh", figsize=(10, max(6, len(communes_for_contact_rate) * 0.32)), color="darkorange"
)
ax.set_yticks(range(len(communes_for_contact_rate)))
ax.set_yticklabels(communes_for_contact_rate["commune_name"].fillna("Unknown commune"))
ax.set(
    title=f"Contact coverage by commune (at least {minimum_commune_prospects} prospects)",
    xlabel="Prospects with phone, email, or website (%)",
    ylabel="Commune",
)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


Flag repeated OSM IDs and non-empty SIRETs for review.


In [ ]:
duplicate_osm_mask = food_prospects["osm_id"].duplicated(keep=False)
has_siret = food_prospects["siret"].notna() & food_prospects["siret"].astype("string").str.strip().ne("")
duplicate_siret_mask = pd.Series(False, index=food_prospects.index)
duplicate_siret_mask.loc[has_siret] = food_prospects.loc[has_siret, "siret"].duplicated(keep=False)

duplicate_summary = pd.Series({
    "rows with repeated OSM ID": duplicate_osm_mask.sum(),
    "repeated OSM IDs": food_prospects.loc[duplicate_osm_mask, "osm_id"].nunique(),
    "rows with repeated non-empty SIRET": duplicate_siret_mask.sum(),
    "repeated non-empty SIRETs": food_prospects.loc[duplicate_siret_mask, "siret"].nunique(),
})
# duplicate_summary and duplicate_siret_examples remain available for review.

duplicate_siret_examples = (
    food_prospects.loc[duplicate_siret_mask, ["siret", "name", "type", "com_nom", "address", "osm_id"]]
    .sort_values("siret")
    .head(20)
)

Summarize prospects by the age of their last OSM update.


In [ ]:
updated_at = pd.to_datetime(food_prospects["last_update"], errors="coerce")
today = pd.Timestamp.today().normalize()
update_age_days = (today - updated_at).dt.days

freshness = pd.Series("unknown or invalid date", index=food_prospects.index, dtype="object")
freshness.loc[update_age_days < 0] = "future-dated"
freshness.loc[update_age_days.between(0, 365)] = "updated within 1 year"
freshness.loc[update_age_days.between(366, 1095)] = "1-3 years old"
freshness.loc[update_age_days > 1095] = "over 3 years old"

freshness_summary = freshness.value_counts().rename_axis("freshness").to_frame("prospects")
freshness_summary["percent"] = (freshness_summary["prospects"] / len(food_prospects) * 100).round(1)
# freshness_summary remains available for review; the chart follows.
print(f"Latest mapped update: {updated_at.max():%Y-%m-%d}")

Plot the age distribution of the last OSM updates.


In [ ]:
import matplotlib.pyplot as plt

freshness_order = ["updated within 1 year", "1-3 years old", "over 3 years old", "unknown or invalid date", "future-dated"]
freshness_counts = freshness.value_counts().reindex(freshness_order, fill_value=0)
ax = freshness_counts[freshness_counts > 0].plot(kind="bar", color="slateblue", figsize=(10, 5))
ax.set(title="Prospect data freshness", xlabel="Age of last mapped update", ylabel="Number of prospects")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

Build and prioritize leads with normalized cuisine and freshness fields.


In [ ]:
lead_columns = [
    "name", "brand", "type", "cuisine_standard", "dish", "com_nom", "address", "phone", "email", "website",
    "opening_hours", "X", "Y", "osm_id", "siret", "last_update",
]
leads = food_prospects[lead_columns].copy()
leads["has_contact"] = has_contact
leads["has_name"] = has_name
leads["last_update"] = pd.to_datetime(leads["last_update"], errors="coerce")
leads["update_age_days"] = (today - leads["last_update"]).dt.days
leads["freshness"] = freshness

leads = leads.sort_values(
    ["has_contact", "has_name", "last_update"],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)

print(f"Prepared lead list: {len(leads):,} prospects")
print(f"With a name and contact detail: {(leads['has_name'] & leads['has_contact']).sum():,}")
lead_preview = leads.head(10).copy()  # Retained for review without rendering raw lead records.


Plot prospect density in the approximate mainland-France window.


In [ ]:
import matplotlib.pyplot as plt

mainland = leads[
    leads["X"].between(-6, 10) & leads["Y"].between(41, 52)
].dropna(subset=["X", "Y"])
print(f"Mapped {len(mainland):,} of {len(leads):,} prospects in the mainland-France view.")

fig, ax = plt.subplots(figsize=(10, 8))
density = ax.hexbin(
    mainland["X"], mainland["Y"], gridsize=90, mincnt=1, bins="log", cmap="magma"
)
ax.set(xlim=(-6, 10), ylim=(41, 52), title="Prospect density — mainland France (approximate geographic window)", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.15)
fig.colorbar(density, ax=ax, label="Prospects per hexagon (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# Export the prioritized lead list as a UTF-8 CSV.
export_path = "france_food_delivery_prospects.csv"
leads.to_csv(export_path, index=False, encoding="utf-8-sig")
print(f"Exported {len(leads):,} prospects to {export_path}")

## 3. Commune market size and cuisine coverage

Compare target food-venue counts with commune population, then examine standardized cuisine coverage and mix. Population density and cuisine counts describe mapped venue supply, not measured delivery demand. The selected INSEE population table excludes Mayotte and does not match every overseas commune code; unmatched places retain venue counts but have no per-resident rate.


Load the compact 2023 total-population lookup for communes; the source dimensions have already been filtered.


In [ ]:
population_file = "insee_commune_population_2023/commune_population_2023.csv"
population_by_commune = pd.read_csv(
    population_file,
    sep=";",
    dtype={"commune_code": "string", "population": "float64"},
)
population_by_commune["commune_code"] = population_by_commune["commune_code"].str.strip()
if population_by_commune["commune_code"].isna().any() or population_by_commune["commune_code"].eq("").any():
    raise ValueError("INSEE population rows contain a blank commune code.")
if population_by_commune["commune_code"].duplicated().any():
    raise ValueError("INSEE total population must have one row per commune code.")

print(f"INSEE communes with 2023 total population: {len(population_by_commune):,}")
print(f"Commune codes retained as text: {population_by_commune['commune_code'].dtype}")
population_row_count = len(population_by_commune)

Join population to target-venue counts by INSEE commune code, report unmatched codes, and compare absolute counts with venues per 10,000 residents.


In [ ]:
commune_market = (
    coverage_data.groupby("commune_code", as_index=False, dropna=False)
    .agg(
        commune_name=("commune_name", "first"),
        target_venues=("osm_id", "nunique"),
    )
)
commune_market = commune_market.merge(
    population_by_commune,
    on="commune_code",
    how="left",
    validate="many_to_one",
)
commune_market["venues_per_10000_residents"] = (
    commune_market["target_venues"] * 10_000
    / commune_market["population"].where(commune_market["population"].gt(0))
)

known_commune_mask = commune_market["commune_code"].ne("Unknown")
matched_population_mask = known_commune_mask & commune_market["population"].notna() & commune_market["population"].gt(0)
unmatched_population = commune_market.loc[
    known_commune_mask & ~matched_population_mask,
    ["commune_code", "commune_name", "target_venues"],
].sort_values("target_venues", ascending=False)
mayotte_mask = commune_market["commune_code"].astype("string").str.startswith("976", na=False)
mayotte_without_population = mayotte_mask & commune_market["population"].isna()

print(
    f"Population matched: {matched_population_mask.sum():,} / "
    f"{known_commune_mask.sum():,} known commune codes"
)
print(f"Prospects in communes without a population match: {int(unmatched_population['target_venues'].sum()):,}")
print(f"Mayotte commune codes without population rates: {int(mayotte_without_population.sum()):,}")
print("Unmatched commune codes (including any unavailable population values):")
unmatched_population_count = len(unmatched_population)

absolute_count_ranking = commune_market.loc[known_commune_mask].nlargest(20, "target_venues")
rate_ranking = commune_market.loc[matched_population_mask].nlargest(20, "venues_per_10000_residents")
market_display_columns = [
    "commune_name", "commune_code", "target_venues", "population", "venues_per_10000_residents"
]
print(f"Top-count communes and population-rate rankings retained for charting/review ({len(absolute_count_ranking)} and {len(rate_ranking)} rows).")

Measure standardized-cuisine coverage against all target prospects in each commune. Split mapped cuisine values only on semicolons and keep dishes separate.


In [ ]:
cuisine_tokens = coverage_data[["osm_id", "commune_code", "commune_name", "cuisine_standard"]].copy()
cuisine_tokens["cuisine_token"] = (
    cuisine_tokens["cuisine_standard"].astype("string").str.split(";")
)
cuisine_tokens = cuisine_tokens.explode("cuisine_token", ignore_index=True)
cuisine_tokens["cuisine_token"] = (
    cuisine_tokens["cuisine_token"].astype("string").str.strip().str.casefold()
)
cuisine_tokens = cuisine_tokens.loc[
    cuisine_tokens["cuisine_token"].notna() & cuisine_tokens["cuisine_token"].ne("")
].drop_duplicates(["osm_id", "commune_code", "cuisine_token"])

cuisine_coverage_by_commune = commune_market[["commune_code", "commune_name", "target_venues"]].copy()
venues_with_standard_cuisine = cuisine_tokens.groupby("commune_code")["osm_id"].nunique()
cuisine_coverage_by_commune["prospects_with_standard_cuisine"] = (
    cuisine_coverage_by_commune["commune_code"].map(venues_with_standard_cuisine).fillna(0).astype("int64")
)
cuisine_coverage_by_commune["standard_cuisine_coverage_pct"] = (
    cuisine_coverage_by_commune["prospects_with_standard_cuisine"]
    / cuisine_coverage_by_commune["target_venues"].where(cuisine_coverage_by_commune["target_venues"].gt(0))
    * 100
).round(1)

known_cuisine_communes = cuisine_coverage_by_commune.loc[
    cuisine_coverage_by_commune["commune_code"].ne("Unknown")
]
print(
    f"Prospects with at least one standardized cuisine: "
    f"{cuisine_tokens['osm_id'].nunique():,} / {food_prospects['osm_id'].nunique():,}"
)
print("Cuisine coverage in the 20 communes with the most target venues:")
top_commune_cuisine_coverage = known_cuisine_communes.nlargest(20, "target_venues").copy()

Show how standardized cuisine coverage and cuisine mix vary across the 20 communes with the most prospects. Shares use cuisine-tagged venues in each commune and are non-exclusive; missing cuisine tags do not show a cuisine is absent.


In [ ]:
cuisine_counts_by_commune = (
    cuisine_tokens.groupby(["commune_code", "cuisine_token"])["osm_id"]
    .nunique()
    .rename("distinct_venues")
    .reset_index()
)
top_cuisine_labels = (
    cuisine_tokens.groupby("cuisine_token")["osm_id"]
    .nunique()
    .nlargest(10)
    .index
)
top_commune_cuisine_rows = known_cuisine_communes.nlargest(20, "target_venues").sort_values("target_venues", ascending=True)
top_cuisine_commune_codes = top_commune_cuisine_rows["commune_code"]
cuisine_mix_counts = (
    cuisine_counts_by_commune.loc[
        cuisine_counts_by_commune["commune_code"].isin(top_cuisine_commune_codes)
        & cuisine_counts_by_commune["cuisine_token"].isin(top_cuisine_labels)
    ]
    .pivot(index="commune_code", columns="cuisine_token", values="distinct_venues")
    .reindex(index=top_cuisine_commune_codes, columns=top_cuisine_labels, fill_value=0)
    .fillna(0)
    .astype("int64")
)
commune_names_by_code = cuisine_coverage_by_commune.set_index("commune_code")["commune_name"]
cuisine_mix_names = commune_names_by_code.reindex(cuisine_mix_counts.index).fillna("Unknown commune")
cuisine_mix_table = cuisine_mix_counts.copy()
cuisine_mix_table.insert(0, "commune_name", cuisine_mix_names.to_numpy())
cuisine_mix_table.insert(0, "commune_code", cuisine_mix_counts.index)

tagged_venue_counts = (
    top_commune_cuisine_rows.set_index("commune_code")["prospects_with_standard_cuisine"]
    .reindex(cuisine_mix_counts.index)
)
commune_cuisine_shares = cuisine_mix_counts.div(tagged_venue_counts.where(tagged_venue_counts.gt(0)), axis=0).mul(100)
commune_cuisine_coverage = (
    top_commune_cuisine_rows.set_index("commune_code")[["prospects_with_standard_cuisine", "standard_cuisine_coverage_pct"]]
    .reindex(cuisine_mix_counts.index)
)

fig, (heat_ax, tagged_ax, coverage_ax) = plt.subplots(
    1, 3, figsize=(17, max(7, len(cuisine_mix_counts) * 0.38)), constrained_layout=True,
    sharey=True, gridspec_kw={"width_ratios": [5.0, 1.7, 1.7], "wspace": 0.12},
)
heatmap = heat_ax.imshow(commune_cuisine_shares.to_numpy(), aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
heat_ax.set_xticks(range(len(commune_cuisine_shares.columns)))
heat_ax.set_xticklabels(commune_cuisine_shares.columns, rotation=45, ha="right")
heat_ax.set_yticks(range(len(commune_cuisine_shares.index)))
heat_ax.set_yticklabels(cuisine_mix_names)
heat_ax.set(title="Cuisine mix", xlabel="Canonical cuisine", ylabel="Commune")
fig.colorbar(heatmap, ax=heat_ax, label="Share of cuisine-tagged venues (%) — non-exclusive")

row_positions = range(len(cuisine_mix_counts.index))
tagged_ax.barh(row_positions, commune_cuisine_coverage["prospects_with_standard_cuisine"], color="#5b8e7d")
tagged_ax.set(title="Cuisine-tagged", xlabel="Venues")
tagged_ax.tick_params(axis="y", left=False, labelleft=False)
tagged_ax.grid(axis="x", alpha=0.25)

coverage_ax.barh(row_positions, commune_cuisine_coverage["standard_cuisine_coverage_pct"], color="#d08c60")
coverage_ax.set(title="Tag coverage", xlabel="All prospects (%)", xlim=(0, 100))
coverage_ax.tick_params(axis="y", left=False, labelleft=False)
coverage_ax.grid(axis="x", alpha=0.25)
heat_ax.invert_yaxis()
fig.suptitle("Cuisine coverage and mix in the 20 largest prospect communes", fontsize=15, fontweight="bold")
# constrained layout leaves room for the figure title and explanatory note.
plt.show()

Compare commune market size with target-venue supply per resident. The plot includes communes with valid 2023 population data, colors each point by cuisine-tag coverage, and highlights the 20 largest prospect markets.


In [ ]:
market_scatter_data = commune_market.merge(
    cuisine_coverage_by_commune[["commune_code", "prospects_with_standard_cuisine", "standard_cuisine_coverage_pct"]],
    on="commune_code",
    how="left",
    validate="one_to_one",
)
population_matched_communes = market_scatter_data.loc[
    market_scatter_data["commune_code"].ne("Unknown")
    & market_scatter_data["population"].gt(0)
].copy()
population_matched_communes["venues_per_10000_residents"] = (
    population_matched_communes["target_venues"] * 10_000 / population_matched_communes["population"]
)
top_market_communes = population_matched_communes.nlargest(20, "target_venues").reset_index(drop=True)
print(
    f"Commune supply comparison: {len(population_matched_communes):,} population-matched communes; "
    f"{int(unmatched_population['target_venues'].sum()):,} prospects lack a usable population rate."
)

fig, (ax, list_ax) = plt.subplots(
    1, 2, figsize=(16, 8), constrained_layout=True,
    gridspec_kw={"width_ratios": [5.5, 1.5], "wspace": 0.08},
)
points = ax.scatter(
    population_matched_communes["venues_per_10000_residents"],
    population_matched_communes["target_venues"],
    c=population_matched_communes["standard_cuisine_coverage_pct"],
    cmap="viridis", s=18, alpha=0.35, linewidths=0,
)
ax.scatter(
    top_market_communes["venues_per_10000_residents"],
    top_market_communes["target_venues"],
    c=top_market_communes["standard_cuisine_coverage_pct"],
    cmap="viridis", vmin=0, vmax=100, s=65, edgecolors="black", linewidths=0.7,
)
for market_rank, row in top_market_communes.iterrows():
    ax.annotate(
        str(market_rank + 1),
        (row["venues_per_10000_residents"], row["target_venues"]),
        xytext=(4, 3), textcoords="offset points", fontsize=7, fontweight="bold",
    )
    list_ax.text(
        0.02, 0.98 - market_rank * 0.047,
        f"{market_rank + 1:>2}. {row['commune_name']}  ({row['target_venues']:,})",
        transform=list_ax.transAxes, va="top", fontsize=8.5,
    )
list_ax.set_title("20 largest markets", fontsize=10)
list_ax.set_axis_off()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    title="Commune prospect markets",
    xlabel="Target venues per 10,000 residents (log scale)",
    ylabel="Target venues (log scale)",
)
ax.grid(alpha=0.2, which="both")
fig.colorbar(points, ax=ax, label="Cuisine-tag coverage (%)")
plt.show()

## 4. Cuisine across France

Compare cuisine tags by the establishment’s French administrative region and compare those cuisines with their mapped regions and countries of origin. Cuisine percentages are non-exclusive because a venue can list multiple cuisines. These are coverage and supply indicators, not measured demand.

Load the compact commune-to-region crosswalk derived from INSEE’s [2026 Code officiel géographique](https://www.insee.fr/fr/information/8740222). It includes metropolitan and overseas administrative regions; territories without a region match will be reported and left out of regional summaries.

In [ ]:
commune_region_file = "commune_region_2026.csv"
commune_region = pd.read_csv(
    commune_region_file,
    dtype={"commune_code": "string", "region_code": "string", "establishment_region": "string"},
)
commune_region["commune_code"] = commune_region["commune_code"].str.strip()
commune_region["region_code"] = commune_region["region_code"].str.strip()
commune_region["establishment_region"] = commune_region["establishment_region"].str.strip()

if commune_region["commune_code"].isna().any() or commune_region["commune_code"].eq("").any():
    raise ValueError("The COG lookup contains a blank commune code.")
if commune_region["commune_code"].duplicated().any():
    raise ValueError("The COG lookup must have one row per commune code.")
if commune_region["establishment_region"].isna().any():
    raise ValueError("Every COG commune code must map to an administrative region.")

overseas_regions = {"Guadeloupe", "Martinique", "Guyane", "La Réunion", "Mayotte"}
missing_overseas = overseas_regions - set(commune_region["establishment_region"])
if missing_overseas:
    raise ValueError(f"COG lookup is missing overseas regions: {sorted(missing_overseas)}")
print(f"COG commune/arrondissement codes: {len(commune_region):,}; unique keys: {commune_region['commune_code'].nunique():,}")
print(f"Administrative regions included: {commune_region['establishment_region'].nunique()} (including all five overseas regions)")
print(f"Crosswalk has {commune_region['establishment_region'].nunique()} mapped regions, including overseas regions.")

Attach each target venue to its establishment region using the text INSEE code. Report codes without a regional match separately, then measure cuisine-tag coverage using all target venues in each mapped region as the denominator.

In [ ]:
region_venues = food_prospects[["osm_id", "com_insee", "cuisine_standard"]].copy()
if region_venues["osm_id"].isna().any() or region_venues["osm_id"].duplicated().any():
    raise ValueError("food_prospects must have one non-missing row per osm_id before regional analysis.")

region_venues["commune_code"] = region_venues["com_insee"].astype("string").str.strip()
region_venues["commune_code"] = region_venues["commune_code"].mask(region_venues["commune_code"].eq(""), pd.NA)
region_venues = region_venues.merge(
    commune_region,
    on="commune_code",
    how="left",
    validate="many_to_one",
    indicator="_region_match",
)
if len(region_venues) != len(food_prospects) or region_venues["osm_id"].nunique() != food_prospects["osm_id"].nunique():
    raise ValueError("The commune-to-region join changed the target-venue row or ID count.")

unmatched_establishment_codes = (
    region_venues.loc[region_venues["establishment_region"].isna()]
    .groupby("commune_code", dropna=False)["osm_id"]
    .nunique()
    .rename("target_venues")
    .reset_index()
    .sort_values("target_venues", ascending=False)
)
print(f"Target venues matched to an administrative region: {region_venues['establishment_region'].notna().sum():,} / {len(region_venues):,}")
print(f"Target venues excluded from regional comparisons (no region match): {int(unmatched_establishment_codes['target_venues'].sum()):,}")
unmatched_establishment_code_count = len(unmatched_establishment_codes)

regional_venues = region_venues.loc[region_venues["establishment_region"].notna()].copy()
regional_target_counts = (
    regional_venues.groupby("establishment_region")["osm_id"]
    .nunique()
    .rename("target_venues")
)
regional_tagged_venue_ids = regional_venues.loc[
    regional_venues["cuisine_standard"].astype("string").str.strip().notna()
    & regional_venues["cuisine_standard"].astype("string").str.strip().ne(""),
    ["establishment_region", "osm_id"],
].drop_duplicates()
regional_tagged_counts = regional_tagged_venue_ids.groupby("establishment_region")["osm_id"].nunique()
regional_cuisine_coverage = pd.DataFrame({"target_venues": regional_target_counts})
regional_cuisine_coverage["venues_with_standard_cuisine"] = regional_tagged_counts.reindex(regional_cuisine_coverage.index).fillna(0).astype("int64")
regional_cuisine_coverage["venues_without_standard_cuisine"] = (
    regional_cuisine_coverage["target_venues"] - regional_cuisine_coverage["venues_with_standard_cuisine"]
)
regional_cuisine_coverage["cuisine_tag_coverage_pct"] = (
    regional_cuisine_coverage["venues_with_standard_cuisine"]
    / regional_cuisine_coverage["target_venues"]
    * 100
).round(1)
print(f"Cuisine-tag coverage is retained for {len(regional_cuisine_coverage)} mapped regions (denominator: all target venues in each region).")

Split `cuisine_standard` only on semicolons, then trim, Unicode-normalize, and casefold each token for matching. Keep underscores and commas intact; keep `dish` out. Map each venue–cuisine pair to its origin region and country while retaining unmatched cuisines for review.

In [ ]:
import unicodedata

regional_cuisine_tokens = region_venues[["osm_id", "establishment_region", "cuisine_standard"]].copy()
regional_cuisine_tokens["canonical_cuisine"] = regional_cuisine_tokens["cuisine_standard"].astype("string").str.split(";")
regional_cuisine_tokens = regional_cuisine_tokens.explode("canonical_cuisine", ignore_index=True)
regional_cuisine_tokens["canonical_cuisine"] = (
    regional_cuisine_tokens["canonical_cuisine"].astype("string").str.normalize("NFKC").str.strip().str.casefold()
)
regional_cuisine_tokens = regional_cuisine_tokens.loc[
    regional_cuisine_tokens["establishment_region"].notna()
    & regional_cuisine_tokens["canonical_cuisine"].notna()
    & regional_cuisine_tokens["canonical_cuisine"].ne(""),
    ["osm_id", "establishment_region", "canonical_cuisine"],
].drop_duplicates(["osm_id", "canonical_cuisine"])

mapping_file = "france_geofabrik_raw_data/cuisine_region_mapping.csv"
cuisine_origin_mapping = pd.read_csv(mapping_file, dtype="string")
required_origin_columns = {"cuisine", "normalized_region", "normalized_country"}
if not required_origin_columns.issubset(cuisine_origin_mapping.columns):
    raise ValueError(f"Cuisine origin mapping must contain {sorted(required_origin_columns)}")
cuisine_origin_mapping["cuisine_match_key"] = (
    cuisine_origin_mapping["cuisine"].map(lambda value: unicodedata.normalize("NFKC", value).strip().casefold() if pd.notna(value) else pd.NA)
)
if cuisine_origin_mapping["cuisine_match_key"].duplicated().any():
    raise ValueError("Cuisine origin mapping contains duplicate normalized cuisine keys.")
cuisine_origin_mapping = cuisine_origin_mapping.rename(columns={
    "normalized_region": "cuisine_origin_region",
    "normalized_country": "cuisine_origin_country",
})[["cuisine_match_key", "cuisine_origin_region", "cuisine_origin_country"]]
regional_cuisine_tokens["cuisine_match_key"] = regional_cuisine_tokens["canonical_cuisine"].map(
    lambda value: unicodedata.normalize("NFKC", value).strip().casefold()
)
venue_cuisine = regional_cuisine_tokens.merge(
    cuisine_origin_mapping,
    on="cuisine_match_key",
    how="left",
    validate="many_to_one",
)
venue_cuisine = venue_cuisine.drop(columns="cuisine_match_key")
if venue_cuisine.duplicated(["osm_id", "canonical_cuisine"]).any():
    raise ValueError("Cuisine-origin mapping multiplied venue–cuisine pairs.")

unmapped_cuisine_origins = (
    venue_cuisine.loc[venue_cuisine["cuisine_origin_country"].isna()]
    .groupby("canonical_cuisine")["osm_id"]
    .nunique()
    .rename("distinct_venues")
    .sort_values(ascending=False)
    .reset_index()
)
# Use valid split tokens for coverage, so empty separator-only strings do not count as tagged.
regional_tagged_venue_ids = venue_cuisine.dropna(subset=["establishment_region"])[
    ["establishment_region", "osm_id"]
].drop_duplicates()
regional_tagged_counts = regional_tagged_venue_ids.groupby("establishment_region")["osm_id"].nunique()
regional_cuisine_coverage["venues_with_standard_cuisine"] = (
    regional_tagged_counts.reindex(regional_cuisine_coverage.index).fillna(0).astype("int64")
)
regional_cuisine_coverage["venues_without_standard_cuisine"] = (
    regional_cuisine_coverage["target_venues"] - regional_cuisine_coverage["venues_with_standard_cuisine"]
)
regional_cuisine_coverage["cuisine_tag_coverage_pct"] = (
    regional_cuisine_coverage["venues_with_standard_cuisine"]
    / regional_cuisine_coverage["target_venues"]
    * 100
).round(1)
origin_mapped_by_region = (
    venue_cuisine.loc[venue_cuisine["cuisine_origin_country"].notna()]
    .groupby("establishment_region")["osm_id"].nunique()
)
regional_cuisine_coverage["venues_with_mapped_cuisine_origin"] = (
    origin_mapped_by_region.reindex(regional_cuisine_coverage.index).fillna(0).astype("int64")
)
regional_cuisine_coverage["origin_mapping_coverage_pct"] = (
    regional_cuisine_coverage["venues_with_mapped_cuisine_origin"]
    / regional_cuisine_coverage["venues_with_standard_cuisine"].where(
        regional_cuisine_coverage["venues_with_standard_cuisine"].gt(0)
    )
    * 100
).round(1)
print(f"Distinct venue–cuisine pairs: {len(venue_cuisine):,}; distinct venues with cuisine tags: {venue_cuisine['osm_id'].nunique():,}")
print(f"Unmapped cuisine-origin labels: {unmapped_cuisine_origins['canonical_cuisine'].nunique():,}; affected venue–cuisine pairs: {int(unmapped_cuisine_origins['distinct_venues'].sum()):,}")
print(f"Cuisine-origin mapping coverage: {origin_mapped_by_region.sum():,} region-tagged venues with a mapped origin; {len(unmapped_cuisine_origins):,} labels remain unmapped.")

Compare cuisine mix across establishment regions with a single heatmap. Color shows the non-exclusive share of cuisine-tagged venues; an aligned coverage column shows cuisine-tag completeness among all target venues. Missing tags do not imply a cuisine is absent.


In [ ]:
regional_cuisine_summary = (
    venue_cuisine.groupby(["establishment_region", "canonical_cuisine"])["osm_id"]
    .nunique()
    .rename("distinct_venues")
    .reset_index()
)
regional_cuisine_summary["cuisine_tagged_venues_in_region"] = regional_cuisine_summary["establishment_region"].map(
    regional_cuisine_coverage["venues_with_standard_cuisine"]
)
regional_cuisine_summary["share_of_cuisine_tagged_venues_pct_nonexclusive"] = (
    regional_cuisine_summary["distinct_venues"]
    / regional_cuisine_summary["cuisine_tagged_venues_in_region"].where(
        regional_cuisine_summary["cuisine_tagged_venues_in_region"].gt(0)
    )
    * 100
).round(1)

top_cuisines_france = (
    regional_cuisine_summary.groupby("canonical_cuisine")["distinct_venues"]
    .sum()
    .nlargest(10)
    .index
)
regional_cuisine_heatmap_counts = (
    regional_cuisine_summary.loc[regional_cuisine_summary["canonical_cuisine"].isin(top_cuisines_france)]
    .pivot(index="establishment_region", columns="canonical_cuisine", values="distinct_venues")
    .fillna(0)
    .astype("int64")
)
regional_cuisine_heatmap_shares = (
    regional_cuisine_summary.loc[regional_cuisine_summary["canonical_cuisine"].isin(top_cuisines_france)]
    .pivot(index="establishment_region", columns="canonical_cuisine", values="share_of_cuisine_tagged_venues_pct_nonexclusive")
    .fillna(0)
)
regional_order = regional_cuisine_coverage.sort_values("target_venues", ascending=True).index
regional_cuisine_heatmap_counts = regional_cuisine_heatmap_counts.reindex(
    index=regional_order, columns=top_cuisines_france, fill_value=0
)
regional_cuisine_heatmap_shares = regional_cuisine_heatmap_shares.reindex(
    index=regional_order, columns=top_cuisines_france, fill_value=0
)

fig, (heat_ax, coverage_ax) = plt.subplots(
    1, 2, figsize=(16, max(7, len(regional_order) * 0.42)), constrained_layout=True,
    sharey=True, gridspec_kw={"width_ratios": [5, 1.8], "wspace": 0.12},
)
heatmap = heat_ax.imshow(regional_cuisine_heatmap_shares.to_numpy(), aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
heat_ax.set_xticks(range(len(regional_cuisine_heatmap_shares.columns)))
heat_ax.set_xticklabels(regional_cuisine_heatmap_shares.columns, rotation=45, ha="right")
heat_ax.set_yticks(range(len(regional_cuisine_heatmap_shares.index)))
heat_ax.set_yticklabels(regional_cuisine_heatmap_shares.index)
heat_ax.set(title="Cuisine mix by establishment region", xlabel="Canonical cuisine", ylabel="Establishment region")
fig.colorbar(heatmap, ax=heat_ax, label="Share of cuisine-tagged venues (%) — non-exclusive")

coverage_values = regional_cuisine_coverage.reindex(regional_order)["cuisine_tag_coverage_pct"]
coverage_ax.barh(range(len(regional_order)), coverage_values, color="#d08c60")
coverage_ax.set(title="Cuisine-tag coverage", xlabel="All target venues (%)", xlim=(0, 100))
coverage_ax.tick_params(axis="y", left=False, labelleft=False)
coverage_ax.grid(axis="x", alpha=0.25)
heat_ax.invert_yaxis()
fig.suptitle("Cuisine mix and tag coverage across France", fontsize=15, fontweight="bold")
# constrained layout leaves room for the title and note.
plt.show()

Show the 10 most common mapped cuisine-origin countries nationally and compare their shares across establishment regions. Shares are non-exclusive and use cuisine-tagged venues as the denominator; the full origin summary dataframes remain available for review.

In [ ]:
origin_rows = venue_cuisine.copy()
origin_rows["cuisine_origin_region"] = origin_rows["cuisine_origin_region"].fillna("Unmapped origin")
origin_rows["cuisine_origin_country"] = origin_rows["cuisine_origin_country"].fillna("Unmapped origin")

regional_origin_region_summary = (
    origin_rows.groupby(["establishment_region", "cuisine_origin_region"])["osm_id"]
    .nunique()
    .rename("distinct_venues")
    .reset_index()
)
regional_origin_country_summary = (
    origin_rows.groupby(["establishment_region", "cuisine_origin_country"])["osm_id"]
    .nunique()
    .rename("distinct_venues")
    .reset_index()
)
for summary in [regional_origin_region_summary, regional_origin_country_summary]:
    summary["cuisine_tagged_venues_in_region"] = summary["establishment_region"].map(
        regional_cuisine_coverage["venues_with_standard_cuisine"]
    )
    summary["share_of_cuisine_tagged_venues_pct_nonexclusive"] = (
        summary["distinct_venues"]
        / summary["cuisine_tagged_venues_in_region"].where(summary["cuisine_tagged_venues_in_region"].gt(0))
        * 100
    ).round(1)

origin_country_counts = (
    venue_cuisine.loc[venue_cuisine["cuisine_origin_country"].notna()]
    .groupby("cuisine_origin_country")["osm_id"]
    .nunique()
    .sort_values(ascending=False)
)
top_origin_countries = origin_country_counts.head(10).index
origin_country_share_heatmap = (
    regional_origin_country_summary.loc[
        regional_origin_country_summary["cuisine_origin_country"].isin(top_origin_countries)
    ]
    .pivot(index="establishment_region", columns="cuisine_origin_country", values="share_of_cuisine_tagged_venues_pct_nonexclusive")
    .fillna(0)
    .reindex(index=regional_cuisine_coverage.sort_values("target_venues", ascending=False).index, columns=top_origin_countries, fill_value=0)
)

fig, (country_ax, origin_heat_ax) = plt.subplots(
    1, 2, figsize=(17, max(7, len(regional_order) * 0.42)), constrained_layout=True,
    gridspec_kw={"width_ratios": [1.25, 3.75], "wspace": 0.35},
)
origin_country_counts.loc[top_origin_countries].sort_values().plot(kind="barh", ax=country_ax, color="#367c8b")
country_ax.set(title="Top origin countries", xlabel="Distinct venues (non-exclusive)", ylabel="Cuisine origin country")
country_ax.grid(axis="x", alpha=0.25)

origin_heatmap = origin_heat_ax.imshow(origin_country_share_heatmap.to_numpy(), aspect="auto", cmap="PuBuGn", vmin=0, vmax=100)
origin_heat_ax.set_xticks(range(len(origin_country_share_heatmap.columns)))
origin_heat_ax.set_xticklabels(origin_country_share_heatmap.columns, rotation=45, ha="right")
origin_heat_ax.set_yticks(range(len(origin_country_share_heatmap.index)))
origin_heat_ax.set_yticklabels(origin_country_share_heatmap.index)
origin_heat_ax.set(title="Origin mix by establishment region", xlabel="Cuisine origin country", ylabel="Establishment region")
fig.colorbar(origin_heatmap, ax=origin_heat_ax, label="Share of cuisine-tagged venues (%) — non-exclusive")
fig.suptitle("Cuisine origins across France", fontsize=15, fontweight="bold")
# constrained layout handles the title, colorbar, and note.
plt.show()